<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_06_transformer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_06 - Transformer (Seq2Seq)**



## **Introducción**

El modelo Transformer es una arquitectura basada en mecanismos de atención, diseñada para modelar dependencias temporales sin recurrencia explícita. A diferencia de RNN, GRU o LSTM, el Transformer procesa la secuencia completa en paralelo y aprende qué instantes del pasado son más relevantes para cada predicción futura.

En el contexto intradía del MNQ, el Transformer resulta especialmente atractivo porque:

- Puede capturar dependencias de largo alcance, incluso cuando la relación entre pasado y futuro no es local.
- Permite ponderar dinámicamente distintos tramos de la ventana histórica mediante atención.
- Escala mejor que modelos recurrentes al aumentar el tamaño de la ventana temporal.

En esta notebook se implementa un Transformer Seq2Seq para predecir la secuencia futura de deltas $ Δ𝑝𝑡𝑠_ℎ $, utilizando una ventana fija de información intradía como entrada.

El foco está puesto en una arquitectura contenida y regularizada, evitando sobre-parametrización, para evaluar si la atención aporta señal adicional frente a modelos recurrentes y convolucionales previamente entrenados.

Regularización principal:

- Early Stopping (obligatorio).
- Dropout en bloques de atención y feed-forward.
- Dimensión del embedding y número de capas limitados.

Este experimento busca responder una pregunta concreta:
¿la atención permite extraer estructura temporal que los modelos GRU, LSTM y TCN no lograron capturar?

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [2]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows/scaled/windows_train_60_z.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows/scaled/windows_valid_60_z.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows/scaled/windows_test_60_z.npz"))
IN_SCALER_60 = Path(os.environ.get("IN_SCALER_60", "data/windows/scaled/scaler_60.pkl"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows/scaled/windows_train_90_z.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows/scaled/windows_valid_90_z.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows/scaled/windows_test_90_z.npz"))
IN_SCALER_90 = Path(os.environ.get("IN_SCALER_90", "data/windows/scaled/scaler_90.pkl"))

#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

#OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [4]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z
IN_SCALER_60 = DRIVE_DIR / IN_SCALER_60

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z
IN_SCALER_90 = DRIVE_DIR / IN_SCALER_90

IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

## **2. Reproducibilidad**

In [5]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [6]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [7]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [8]:
# Construye un StageConfig leyendo ambos reports.
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )


In [9]:
states_h60 = load_state_from_reports(horizon=60)
states_h60

StageConfig(horizon=60, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_60'], target_name='delta_pts_60', delta_base=52.12, delta_op=84.18)

In [10]:
states_h90 = load_state_from_reports(horizon=90)
states_h90

StageConfig(horizon=90, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_30'], target_name='delta_pts_90', delta_base=60.75, delta_op=97.22)


## **4. Importar métricas comunes desde .py**

In [11]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2seq_metrics import compute_seq2seq_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [12]:
print(compute_seq2seq_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2seq.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    y_pred : np.ndarray
        Valores predichos con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    compute_r2 : bool
        Si True, calcula R² sobre la secuencia completa concatenada.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 (y_true_last==0 o y_pred_last==0)
        al calcular DA_last. Esto evita ambigüedad en la dirección.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales y diagnóstico por paso.
    


## **5. Carga de data windows**

In [13]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [14]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [15]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER_60
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER_90

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [16]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [17]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (912, 29, 7) (912, 29)
H60 Valid: (195, 29, 7) (195, 29)
H60 Test : (196, 29, 7) (196, 29)
H90 Train: (912, 29, 7) (912, 29)
H90 Valid: (195, 29, 7) (195, 29)
H90 Test : (196, 29, 7) (196, 29)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **6. Sanity Check**

In [18]:
def sanity_check(
    X: np.ndarray,                 # Tensores de entrada: (n_samples, seq_len, n_features)
    y: np.ndarray,                 # Targets: (n_samples, seq_len) o (n_samples, seq_len, 1)
    name: str,                     # Nombre lógico del split (ej: "train_h60", "valid_h90")
    *,
    expected_seq_len: int,         # Largo de secuencia esperado (ej: 29)
    expected_n_features: int,      # Número de features esperado (ej: 8)
) -> None:
    # --------------------------------------------------
    # Chequeos de dimensionalidad
    # --------------------------------------------------

    # X debe ser estrictamente 3D: (muestras, tiempo, features)
    assert X.ndim == 3, (
        f"{name}: X debe ser 3D (n, seq, feat)"
    )

    # y puede ser 2D (n, seq) o 3D (n, seq, 1)
    assert y.ndim in (2, 3), (
        f"{name}: y debe ser 2D o 3D (n, seq) o (n, seq, 1)"
    )

    # --------------------------------------------------
    # Chequeos de consistencia temporal y estructural
    # --------------------------------------------------

    # Verifica que el largo temporal de X coincida con el esperado
    assert X.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en X: {X.shape[1]} != {expected_seq_len}"
    )

    # Verifica que la cantidad de features en X sea la esperada
    assert X.shape[2] == expected_n_features, (
        f"{name}: n_features inesperado en X: {X.shape[2]} != {expected_n_features}"
    )

    # Verifica que y tenga el mismo largo temporal que X
    assert y.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en y: {y.shape[1]} != {expected_seq_len}"
    )

    # --------------------------------------------------
    # Chequeos numéricos (sanidad de valores)
    # --------------------------------------------------

    # Asegura que X no contenga NaN ni infinitos
    assert np.isfinite(X).all(), (
        f"{name}: X contiene NaN/inf"
    )

    # Asegura que y no contenga NaN ni infinitos
    assert np.isfinite(y).all(), (
        f"{name}: y contiene NaN/inf"
    )

In [19]:
from __future__ import annotations

from typing import Any, Dict
import numpy as np


def run_sanity_checks_for_bundle(bundle: Dict[str, Any], *, tag: str) -> None:
    """
    Ejecuta sanity_check para train/valid/test usando únicamente variables locales.

    Espera un bundle con estructura:
    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
      ...
    }
    """
    # Extrae arrays localmente (no crea globals)
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]

    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]

    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    # Toma expected_* desde TRAIN (consistencia)
    expected_seq_len = int(X_tr.shape[1])
    expected_n_features = int(X_tr.shape[2])

    # Ejecuta sanity checks por split
    sanity_check(X_tr, y_tr, f"train_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_va, y_va, f"valid_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_te, y_te, f"test_{tag}",  expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)

    # Mensaje de OK por bundle/horizonte
    h = bundle.get("horizon", "NA")
    print(f"OK {tag} (h={h}) | seq_len={expected_seq_len} | n_features={expected_n_features}")

    # Opcional: borra referencias locales explícitamente (no es estrictamente necesario)
    del X_tr, y_tr, X_va, y_va, X_te, y_te


def run_sanity_checks_all_horizons(bundle_60: Dict[str, Any], bundle_90: Dict[str, Any]) -> None:
    """Corre sanity checks para ambos horizontes."""
    run_sanity_checks_for_bundle(bundle_60, tag="h60")
    run_sanity_checks_for_bundle(bundle_90, tag="h90")

In [20]:
run_sanity_checks_all_horizons(bundle_60, bundle_90)

OK h60 (h=60) | seq_len=29 | n_features=7
OK h90 (h=90) | seq_len=29 | n_features=7


## **7. Definición del modelo — placeholder**

Idea mínima antes del código (para que sepas qué mirar)
- Entrada: (N, 29, 7)
- Salida: (N, 29, 1) (seq2seq directo)
- Bloques TCN:
  - Convolución causal
  - Dilatación creciente (1, 2, 4, …)
  - Residual connection
- Sin recurrencia, sin encoder/decoder.

### **7.1. Definición de positional encoding**

In [21]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

def build_sinusoidal_positional_encoding(seq_len: int, d_model: int) -> np.ndarray:
    """
    Positional encoding sinusoidal (estándar).
    Retorna shape: (1, seq_len, d_model) para sumarlo al embedding.
    """
    pos = np.arange(seq_len)[:, None]          # (seq_len, 1)
    i = np.arange(d_model)[None, :]            # (1, d_model)

    # Frecuencias por dimensión
    angle_rates = 1.0 / np.power(10000.0, (2 * (i // 2)) / np.float32(d_model))
    angles = pos * angle_rates                 # (seq_len, d_model)

    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    pe[:, 0::2] = np.sin(angles[:, 0::2])      # posiciones pares
    pe[:, 1::2] = np.cos(angles[:, 1::2])      # posiciones impares

    return pe[None, :, :]                      # (1, seq_len, d_model)


class AddPositionalEncoding(layers.Layer):
    """
    Capa Keras que suma positional encoding (constante, no entrenable)
    a un tensor (batch, seq_len, d_model).
    """
    def __init__(self, seq_len: int, d_model: int, name="pos_enc", **kwargs):
        super().__init__(name=name, **kwargs)
        pe = build_sinusoidal_positional_encoding(seq_len, d_model)
        self.pe = keras.backend.constant(pe)   # constante TF (no entrenable)

    def call(self, x):
        return x + self.pe


### **7.2. Definición del bloque Transformer Encoder (MultiheadAttention + FFN)**

In [22]:
from tensorflow.keras import layers

def transformer_encoder_block(
    x,
    *,
    d_model: int,
    num_heads: int,
    ffn_dim: int,
    dropout: float = 0.10,
    name: str = "enc_block",
):
    """
    Un bloque Transformer Encoder (clásico) para series temporales.

    Entrada:
      x: tensor (batch, seq_len, d_model)

    Componentes:
      1) Self-Attention (MultiHeadAttention) + residual + LayerNorm
      2) Feed-Forward (2 Dense)             + residual + LayerNorm

    Regularización:
      - Dropout en la salida de atención y dentro del FFN
      - No hay recurrencia, así que no existe recurrent_dropout
    """
    # Validación básica: d_model debe ser divisible por num_heads
    if d_model % num_heads != 0:
        raise ValueError("d_model debe ser divisible por num_heads.")

    # ============================================================
    # 1) Self-Attention (query=key=value=x)
    # ============================================================
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,   # dimensión por head
        dropout=dropout,
        name=f"{name}_mha",
    )(x, x)

    # Dropout + Residual + LayerNorm
    attn = layers.Dropout(dropout, name=f"{name}_attn_dropout")(attn)
    x = layers.Add(name=f"{name}_attn_residual")([x, attn])
    x = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_attn_ln")(x)

    # ============================================================
    # 2) Feed-Forward Network (FFN)
    # ============================================================
    ffn = layers.Dense(ffn_dim, activation="relu", name=f"{name}_ffn_dense1")(x)
    ffn = layers.Dropout(dropout, name=f"{name}_ffn_dropout1")(ffn)
    ffn = layers.Dense(d_model, activation="linear", name=f"{name}_ffn_dense2")(ffn)
    ffn = layers.Dropout(dropout, name=f"{name}_ffn_dropout2")(ffn)

    # Residual + LayerNorm
    x = layers.Add(name=f"{name}_ffn_residual")([x, ffn])
    x = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ffn_ln")(x)

    return x


### **7.3. Armado de modelo completo**

In [23]:
from tensorflow import keras
from tensorflow.keras import layers

def build_transformer_seq2seq_direct(
    *,
    window_len: int = 29,
    n_features: int = 7,
    d_model: int = 64,
    num_heads: int = 4,
    ffn_dim: int = 128,
    num_layers: int = 2,
    dropout: float = 0.10,
    lr: float = 1e-3,
):
    """
    Transformer Seq2Seq directo (encoder-only):
    - Input : (window_len, n_features)
    - Output: (window_len, 1)  # en su setup, horizon_len=window_len=29
    """

    # -------------------------
    # Input
    # -------------------------
    x_in = keras.Input(shape=(window_len, n_features), name="x_in")

    # -------------------------
    # Proyección a embedding (d_model)
    # -------------------------
    x = layers.Dense(d_model, activation="linear", name="feat_proj")(x_in)

    # -------------------------
    # Positional Encoding (Paso 1)
    # -------------------------
    x = AddPositionalEncoding(seq_len=window_len, d_model=d_model, name="pos_enc")(x)
    x = layers.Dropout(dropout, name="emb_dropout")(x)

    # -------------------------
    # Encoder blocks (Paso 2)
    # -------------------------
    for i in range(1, num_layers + 1):
        x = transformer_encoder_block(
            x,
            d_model=d_model,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            dropout=dropout,
            name=f"enc_{i}",
        )

    # -------------------------
    # Salida por timestep: (batch, window_len, 1)
    # -------------------------
    y_hat = layers.Dense(1, activation="linear", name="y_hat")(x)

    model = keras.Model(inputs=x_in, outputs=y_hat, name="Transformer_Seq2Seq_Direct")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=[keras.metrics.MAE],
    )

    return model


### **7.4. Construcción de modelo desde bundles**

In [27]:
def to_3d_y(y: np.ndarray) -> np.ndarray:
    """
    Normaliza y a 3D para Seq2Seq:
      - (N, H)    -> (N, H, 1)
      - (N, H, 1) -> (N, H, 1)

    Esto es consistente con la salida del GRU Seq2Seq:
      y_hat: (N, H, 1)
    """
    # Caso: y ya es (N, H, 1)
    if y.ndim == 3:
        if y.shape[-1] != 1:
            raise ValueError(f"Se esperaba y con último dim=1, got {y.shape}")
        return y

    # Caso: y viene como (N, H) y lo expandimos a (N, H, 1)
    if y.ndim == 2:
        return y[..., None]

    # Cualquier otra forma no es compatible con este pipeline
    raise ValueError(f"Forma inesperada para y: {y.shape}")

In [28]:
# ---------------------------------------------
# PASO 4) build_model_for_bundle (Transformer) – por horizonte (H60 / H90)
# ---------------------------------------------
# Objetivo:
# - Construir 2 modelos independientes (uno para H60 y otro para H90)
# - Inferir window_len, n_features y horizon_len desde TRAIN del bundle
# - Mantener el mismo esquema que venimos usando en MLP/GRU/LSTM/TCN
#
# Nota:
# - Este Transformer "directo" asume horizon_len == window_len (su caso: 29 y 29).
# - Esto NO entrena nada, solo construye y compila el modelo.
# ---------------------------------------------

def build_transformer_model_for_bundle(bundle_h: dict):
    """
    Construye un Transformer Seq2Seq directo para un bundle (H60 o H90),
    inferiendo shapes desde TRAIN.

    Retorna:
      keras.Model listo para fit().
    """
    # -------------------------
    # Leer X/y desde TRAIN (para inferir shapes)
    # -------------------------
    X_train = bundle_h["train"]["X"]  # (N, window_len, n_features)
    y_train = bundle_h["train"]["y"]  # (N, H) o (N, H, 1)

    window_len = int(X_train.shape[1])
    n_features = int(X_train.shape[2])

    # Normalizamos y_train a 3D para inferir horizon_len robustamente
    y_train_3d = to_3d_y(y_train)     # (N, H, 1)
    horizon_len = int(y_train_3d.shape[1])

    # -------------------------
    # Validación clave del setup actual
    # -------------------------
    # En este Transformer directo, la salida es (window_len, 1),
    # por lo que horizon_len debe coincidir con window_len.
    if horizon_len != window_len:
        raise ValueError(
            f"Transformer directo requiere horizon_len == window_len. "
            f"Got horizon_len={horizon_len} vs window_len={window_len}."
        )

    # -------------------------
    # Construcción del modelo (parámetros moderados)
    # -------------------------
    model = build_transformer_seq2seq_direct(
        window_len=window_len,
        n_features=n_features,
        d_model=64,       # embedding moderado
        num_heads=4,      # pocas heads
        ffn_dim=128,      # FFN moderado
        num_layers=2,     # 1–2 capas para empezar
        dropout=0.10,     # dropout leve
        lr=1e-3,
    )

    return model






In [29]:
# ---------------------------------------------
# Ejemplo de uso (H60 / H90) – solo construcción
# ---------------------------------------------
model_tr_60 = build_transformer_model_for_bundle(bundle_60)
model_tr_90 = build_transformer_model_for_bundle(bundle_90)
model_tr_60.summary()
model_tr_90.summary()

Model: "Transformer_Seq2Seq_Direct"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x_in (InputLayer)   │ (None, 29, 7)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_proj (Dense)   │ (None, 29, 64)    │        512 │ x_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_enc             │ (None, 29, 64)    │          0 │ feat_proj[0][0]   │
│ (AddPositionalEnco… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_dropout         │ (None, 29, 64)    │          0 │ pos_enc[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_mha           │ (None, 29, 64)    │     16,640 │ emb_dropout[0][0… │
│ (MultiHeadAttentio… │                   │            │ emb_dropout[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_attn_dropout  │ (None, 29, 64)    │          0 │ enc_1_mha[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_attn_residual │ (None, 29, 64)    │          0 │ emb_dropout[0][0… │
│ (Add)               │                   │            │ enc_1_attn_dropo… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_attn_ln       │ (None, 29, 64)    │        128 │ enc_1_attn_resid… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dense1    │ (None, 29, 128)   │      8,320 │ enc_1_attn_ln[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dropout1  │ (None, 29, 128)   │          0 │ enc_1_ffn_dense1… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dense2    │ (None, 29, 64)    │      8,256 │ enc_1_ffn_dropou… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dropout2  │ (None, 29, 64)    │          0 │ enc_1_ffn_dense2… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_residual  │ (None, 29, 64)    │          0 │ enc_1_attn_ln[0]… │
│ (Add)               │                   │            │ enc_1_ffn_dropou… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_ln        │ (None, 29, 64)    │        128 │ enc_1_ffn_residu… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_2_mha           │ (None, 29, 64)    │     16,640 │ enc_1_ffn_ln[0][… │
│ (MultiHeadAttentio… │                   │            │ enc_1_ffn_ln[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_2_attn_dropout  │ (None, 29, 64)    │          0 │ enc_2_mha[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_2_attn_residual │ (None, 29, 64)    │          0 │ enc_1_ffn_ln[0][… │
│ (Add)               │                   │            │ enc_2_attn_dropo… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 67,521 (263.75 KB)

 Trainable params: 67,521 (263.75 KB)

 Non-trainable params: 0 (0.00 B)

Model: "Transformer_Seq2Seq_Direct"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x_in (InputLayer)   │ (None, 29, 7)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_proj (Dense)   │ (None, 29, 64)    │        512 │ x_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_enc             │ (None, 29, 64)    │          0 │ feat_proj[0][0]   │
│ (AddPositionalEnco… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_dropout         │ (None, 29, 64)    │          0 │ pos_enc[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_mha           │ (None, 29, 64)    │     16,640 │ emb_dropout[0][0… │
│ (MultiHeadAttentio… │                   │            │ emb_dropout[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_attn_dropout  │ (None, 29, 64)    │          0 │ enc_1_mha[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_attn_residual │ (None, 29, 64)    │          0 │ emb_dropout[0][0… │
│ (Add)               │                   │            │ enc_1_attn_dropo… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_attn_ln       │ (None, 29, 64)    │        128 │ enc_1_attn_resid… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dense1    │ (None, 29, 128)   │      8,320 │ enc_1_attn_ln[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dropout1  │ (None, 29, 128)   │          0 │ enc_1_ffn_dense1… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dense2    │ (None, 29, 64)    │      8,256 │ enc_1_ffn_dropou… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_dropout2  │ (None, 29, 64)    │          0 │ enc_1_ffn_dense2… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_residual  │ (None, 29, 64)    │          0 │ enc_1_attn_ln[0]… │
│ (Add)               │                   │            │ enc_1_ffn_dropou… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_1_ffn_ln        │ (None, 29, 64)    │        128 │ enc_1_ffn_residu… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_2_mha           │ (None, 29, 64)    │     16,640 │ enc_1_ffn_ln[0][… │
│ (MultiHeadAttentio… │                   │            │ enc_1_ffn_ln[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_2_attn_dropout  │ (None, 29, 64)    │          0 │ enc_2_mha[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_2_attn_residual │ (None, 29, 64)    │          0 │ enc_1_ffn_ln[0][… │
│ (Add)               │                   │            │ enc_2_attn_dropo… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 67,521 (263.75 KB)

 Trainable params: 67,521 (263.75 KB)

 Non-trainable params: 0 (0.00 B)

### **7.4. Verificación de shapes**

In [30]:
# ---------------------------------------------
# PASO 1) Verificación de shapes para Seq2Seq (TCN / GRU / LSTM)
# ---------------------------------------------
# Objetivo:
# - Confirmar que los bundles H60 y H90 tienen shapes consistentes
# - Obtener (window_len, n_features, horizon_len) desde TRAIN
# - Verificar que VALID mantiene los mismos (window_len, n_features, horizon_len)
#
# Nota:
# - Esto NO entrena nada. Solo inspecciona y valida dimensiones.
# ---------------------------------------------

def assert_same(a: int, b: int, *, msg: str) -> None:
    """Levanta error con mensaje claro si dos valores no coinciden."""
    if int(a) != int(b):
        raise ValueError(f"{msg} (train={a} vs valid={b})")


def check_bundle_shapes_for_seq2seq(bundle_h: dict, *, tag: str) -> dict:
    """
    Chequea shapes relevantes para un bundle (por ejemplo H60 o H90).

    Retorna un dict con:
      - window_len
      - n_features
      - horizon_len
      - train_shapes (X,y)
      - valid_shapes (X,y)
    """
    # -------------------------
    # TRAIN: shapes base
    # -------------------------
    X_train = bundle_h["train"]["X"]   # (N, window_len, n_features)
    y_train = bundle_h["train"]["y"]   # (N, H) o (N, H, 1)

    window_len_train = int(X_train.shape[1])
    n_features_train = int(X_train.shape[2])

    # Normalizamos y_train a 3D para leer horizon_len de forma uniforme
    y_train_3d = to_3d_y(y_train)      # (N, H, 1)
    horizon_len_train = int(y_train_3d.shape[1])

    # -------------------------
    # VALID: debe ser consistente con TRAIN
    # -------------------------
    X_valid = bundle_h["valid"]["X"]
    y_valid = bundle_h["valid"]["y"]

    window_len_valid = int(X_valid.shape[1])
    n_features_valid = int(X_valid.shape[2])

    y_valid_3d = to_3d_y(y_valid)
    horizon_len_valid = int(y_valid_3d.shape[1])

    # -------------------------
    # Validaciones de consistencia
    # -------------------------
    assert_same(window_len_train, window_len_valid, msg=f"[{tag}] window_len inconsistente")
    assert_same(n_features_train, n_features_valid, msg=f"[{tag}] n_features inconsistente")
    assert_same(horizon_len_train, horizon_len_valid, msg=f"[{tag}] horizon_len inconsistente")

    # -------------------------
    # Reporte compacto
    # -------------------------
    info = {
        "tag": tag,
        "window_len": window_len_train,
        "n_features": n_features_train,
        "horizon_len": horizon_len_train,
        "train_shapes": {
            "X": tuple(X_train.shape),
            "y_raw": tuple(y_train.shape),
            "y_3d": tuple(y_train_3d.shape),
        },
        "valid_shapes": {
            "X": tuple(X_valid.shape),
            "y_raw": tuple(y_valid.shape),
            "y_3d": tuple(y_valid_3d.shape),
        },
    }

    return info


# -------------------------
# Ejecutar checks para H60 y H90
# -------------------------
info_60 = check_bundle_shapes_for_seq2seq(bundle_60, tag="H60")
info_90 = check_bundle_shapes_for_seq2seq(bundle_90, tag="H90")

# Imprimir resumen legible
print(
    f"[H60] window_len={info_60['window_len']} | n_features={info_60['n_features']} | horizon_len={info_60['horizon_len']}\n"
    f"      train X={info_60['train_shapes']['X']} y_raw={info_60['train_shapes']['y_raw']} y_3d={info_60['train_shapes']['y_3d']}\n"
    f"      valid X={info_60['valid_shapes']['X']} y_raw={info_60['valid_shapes']['y_raw']} y_3d={info_60['valid_shapes']['y_3d']}\n"
)

print(
    f"[H90] window_len={info_90['window_len']} | n_features={info_90['n_features']} | horizon_len={info_90['horizon_len']}\n"
    f"      train X={info_90['train_shapes']['X']} y_raw={info_90['train_shapes']['y_raw']} y_3d={info_90['train_shapes']['y_3d']}\n"
    f"      valid X={info_90['valid_shapes']['X']} y_raw={info_90['valid_shapes']['y_raw']} y_3d={info_90['valid_shapes']['y_3d']}\n"
)


[H60] window_len=29 | n_features=7 | horizon_len=29
      train X=(912, 29, 7) y_raw=(912, 29) y_3d=(912, 29, 1)
      valid X=(195, 29, 7) y_raw=(195, 29) y_3d=(195, 29, 1)

[H90] window_len=29 | n_features=7 | horizon_len=29
      train X=(912, 29, 7) y_raw=(912, 29) y_3d=(912, 29, 1)
      valid X=(195, 29, 7) y_raw=(195, 29) y_3d=(195, 29, 1)



### **7.5. Construcción de modelo por horizontes**

In [ ]:
K.clear_session()  # evita reutilizar tensores/modelos anteriores

# Construye modelos
model_tcn_60 = build_tcn_model_for_bundle(bundle_60)
model_tcn_90 = build_tcn_model_for_bundle(bundle_90)

# Resúmenes para verificar arquitectura y shapes
print("\n" + "="*80)
print("MODEL SUMMARY - TCN Seq2Seq (H60)")
print("="*80)
model_tcn_60.summary()

print("\n" + "="*80)
print("MODEL SUMMARY - TCN Seq2Seq (H90)")
print("="*80)
model_tcn_90.summary()

print("\n" + "="*80)
print("SHAPES CHECK")
print("="*80)
print("H60 input:", model_tcn_60.input.shape, "output:", model_tcn_60.output.shape)
print("H90 input:", model_tcn_90.input.shape, "output:", model_tcn_90.output.shape)


MODEL SUMMARY - TCN Seq2Seq (H60)


Model: "TCN_Seq2Seq"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tcn_in (InputLayer) │ (None, 29, 7)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_conv1   │ (None, 29, 64)    │      1,408 │ tcn_in[0][0]      │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_dropou… │ (None, 29, 64)    │          0 │ tcn_block_1_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_conv2   │ (None, 29, 64)    │     12,352 │ tcn_block_1_drop… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_dropou… │ (None, 29, 64)    │          0 │ tcn_block_1_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_residu… │ (None, 29, 64)    │        512 │ tcn_in[0][0]      │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_add     │ (None, 29, 64)    │          0 │ tcn_block_1_drop… │
│ (Add)               │                   │            │ tcn_block_1_resi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_conv1   │ (None, 29, 64)    │     12,352 │ tcn_block_1_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_dropou… │ (None, 29, 64)    │          0 │ tcn_block_2_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_conv2   │ (None, 29, 64)    │     12,352 │ tcn_block_2_drop… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_dropou… │ (None, 29, 64)    │          0 │ tcn_block_2_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_add     │ (None, 29, 64)    │          0 │ tcn_block_2_drop… │
│ (Add)               │                   │            │ tcn_block_1_add[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_conv1   │ (None, 29, 64)    │     12,352 │ tcn_block_2_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_dropou… │ (None, 29, 64)    │          0 │ tcn_block_3_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_conv2   │ (None, 29, 64)    │     12,352 │ tcn_block_3_drop… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_dropou… │ (None, 29, 64)    │          0 │ tcn_block_3_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_add     │ (None, 29, 64)    │          0 │ tcn_block_3_drop… │
│ (Add)               │                   │            │ tcn_block_2_add[

 Total params: 63,745 (249.00 KB)

 Trainable params: 63,745 (249.00 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tcn_in (InputLayer) │ (None, 29, 7)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_conv1   │ (None, 29, 64)    │      1,408 │ tcn_in[0][0]      │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_dropou… │ (None, 29, 64)    │          0 │ tcn_block_1_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_conv2   │ (None, 29, 64)    │     12,352 │ tcn_block_1_drop… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_dropou… │ (None, 29, 64)    │          0 │ tcn_block_1_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_residu… │ (None, 29, 64)    │        512 │ tcn_in[0][0]      │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_add     │ (None, 29, 64)    │          0 │ tcn_block_1_drop… │
│ (Add)               │                   │            │ tcn_block_1_resi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_conv1   │ (None, 29, 64)    │     12,352 │ tcn_block_1_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_dropou… │ (None, 29, 64)    │          0 │ tcn_block_2_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_conv2   │ (None, 29, 64)    │     12,352 │ tcn_block_2_drop… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_dropou… │ (None, 29, 64)    │          0 │ tcn_block_2_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_add     │ (None, 29, 64)    │          0 │ tcn_block_2_drop… │
│ (Add)               │                   │            │ tcn_block_1_add[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_conv1   │ (None, 29, 64)    │     12,352 │ tcn_block_2_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_dropou… │ (None, 29, 64)    │          0 │ tcn_block_3_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_conv2   │ (None, 29, 64)    │     12,352 │ tcn_block_3_drop… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_dropou… │ (None, 29, 64)    │          0 │ tcn_block_3_conv… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_add     │ (None, 29, 64)    │          0 │ tcn_block_3_drop… │
│ (Add)               │                   │            │ tcn_block_2_add[

 Total params: 63,745 (249.00 KB)

 Trainable params: 63,745 (249.00 KB)

 Non-trainable params: 0 (0.00 B)


SHAPES CHECK
H60 input: (None, 29, 7) output: (None, 29, 1)
H90 input: (None, 29, 7) output: (None, 29, 1)


In [ ]:
# (Opcional) Confirmar que está en eager
print("H60 run_eagerly:", model_tcn_60.run_eagerly)
print("H90 run_eagerly:", model_tcn_90.run_eagerly)

H60 run_eagerly: False
H90 run_eagerly: False


In [ ]:
print("H60 inputs:", [t.shape for t in model_tcn_60.inputs], "output:", model_tcn_60.output.shape)
print("H90 inputs:", [t.shape for t in model_tcn_90.inputs], "output:", model_tcn_90.output.shape)

H60 inputs: [(None, 29, 7)] output: (None, 29, 1)
H90 inputs: [(None, 29, 7)] output: (None, 29, 1)


### **7.6. Preparar X/y para TRAIN y VALID**

In [31]:
# ---------------------------------------------
# Preparar X/y para TRAIN y VALID (Transformer)
# ---------------------------------------------
# Objetivo:
# - Armar exactamente lo que espera model.fit(...) y model.predict(...)
# - Para cada horizonte:
#     X_train: (N_train, 29, 7)
#     y_train: (N_train, 29, 1)
#     X_valid: (N_valid, 29, 7)
#     y_valid: (N_valid, 29, 1)
# ---------------------------------------------

def get_seq2seq_xy_from_bundle(bundle_h: dict, *, split: str):
    """
    Helper genérico para modelos RepeatVector (GRU/LSTM):
    - X: (N, window_len, n_features)
    - y: (N, horizon_len, 1)
    """
    X = bundle_h[split]["X"]
    y = to_3d_y(bundle_h[split]["y"])
    return X, y

In [32]:
# -------------------------
# H60: TRAIN y VALID
# -------------------------
X_train_60, y_train_60_3d = get_seq2seq_xy_from_bundle(bundle_60, split="train")
X_valid_60, y_valid_60_3d = get_seq2seq_xy_from_bundle(bundle_60, split="valid")

print("\n" + "=" * 80)
print("H60 - DATA CHECK (Transformer)")
print("=" * 80)
print("X_train:", X_train_60.shape, "y_train:", y_train_60_3d.shape)
print("X_valid:", X_valid_60.shape, "y_valid:", y_valid_60_3d.shape)


# -------------------------
# H90: TRAIN y VALID
# -------------------------
X_train_90, y_train_90_3d = get_seq2seq_xy_from_bundle(bundle_90, split="train")
X_valid_90, y_valid_90_3d = get_seq2seq_xy_from_bundle(bundle_90, split="valid")

print("\n" + "=" * 80)
print("H90 - DATA CHECK (Transformer)")
print("=" * 80)
print("X_train:", X_train_90.shape, "y_train:", y_train_90_3d.shape)
print("X_valid:", X_valid_90.shape, "y_valid:", y_valid_90_3d.shape)


H60 - DATA CHECK (Transformer)
X_train: (912, 29, 7) y_train: (912, 29, 1)
X_valid: (195, 29, 7) y_valid: (195, 29, 1)

H90 - DATA CHECK (Transformer)
X_train: (912, 29, 7) y_train: (912, 29, 1)
X_valid: (195, 29, 7) y_valid: (195, 29, 1)


### **7.7. Entrenamiento Transformer**

In [33]:
# ============================================================
# PASO 4) FIT ROBUSTO – Transformer Seq2Seq (VALID ONLY)
# ============================================================
# Este bloque:
# - Verifica GPU disponible
# - Verifica shapes críticos antes de entrenar
# - Entrena H60 y H90 con EarlyStopping como regularización principal
# - Usa SOLO TRAIN + VALID (no test)
# ============================================================

import tensorflow as tf
from tensorflow import keras

# -------------------------
# 1) Verificación de GPU
# -------------------------
gpus = tf.config.list_logical_devices("GPU")
device_name = "/device:GPU:0" if gpus else "/device:CPU:0"
print("Dispositivo a usar:", device_name)

# (Opcional) mejorar performance GPU en TF
try:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: activado (mixed_float16)")
except Exception:
    print("Mixed precision: no disponible / no aplicado")

# -------------------------
# 2) Chequeo de shapes (seguro antes de fit)
# -------------------------
def check_xy_shapes(X_train, y_train, X_valid, y_valid, tag: str):
    """
    Verifica consistencia básica de shapes para:
      fit(X_train, y_train) y validation_data=(X_valid, y_valid)
    """
    # X debe ser 3D: (N, window_len, n_features)
    assert X_train.ndim == 3, f"[{tag}] X_train no es 3D"
    assert X_valid.ndim == 3, f"[{tag}] X_valid no es 3D"

    # y debe ser 3D: (N, horizon_len, 1)
    assert y_train.ndim == 3, f"[{tag}] y_train no es 3D"
    assert y_valid.ndim == 3, f"[{tag}] y_valid no es 3D"
    assert y_train.shape[-1] == 1, f"[{tag}] y_train último dim != 1"
    assert y_valid.shape[-1] == 1, f"[{tag}] y_valid último dim != 1"

    # N consistente
    assert X_train.shape[0] == y_train.shape[0], f"[{tag}] N_train inconsistente"
    assert X_valid.shape[0] == y_valid.shape[0], f"[{tag}] N_valid inconsistente"

    # horizon_len consistente entre train y valid
    assert y_train.shape[1] == y_valid.shape[1], f"[{tag}] horizon_len train/valid inconsistente"

    print(f"[{tag}] Shapes OK:",
          "X_train", X_train.shape, "y_train", y_train.shape,
          "| X_valid", X_valid.shape, "y_valid", y_valid.shape)

check_xy_shapes(X_train_60, y_train_60_3d, X_valid_60, y_valid_60_3d, tag="H60")
check_xy_shapes(X_train_90, y_train_90_3d, X_valid_90, y_valid_90_3d, tag="H90")

# -------------------------
# 3) Callbacks (regularización)
# -------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1,
)

# -------------------------
# 4) FIT en GPU (H60 y H90)
# -------------------------
with tf.device(device_name):

    print("\n" + "=" * 80)
    print("FIT – Transformer H60 (VALID ONLY)")
    print("=" * 80)

    hist_tr_60 = model_tr_60.fit(
        X_train_60,
        y_train_60_3d,
        validation_data=(X_valid_60, y_valid_60_3d),
        epochs=200,
        batch_size=256,
        callbacks=[early_stop, reduce_lr],
        verbose=1,
    )

    print("\n" + "=" * 80)
    print("FIT – Transformer H90 (VALID ONLY)")
    print("=" * 80)

    hist_tr_90 = model_tr_90.fit(
        X_train_90,
        y_train_90_3d,
        validation_data=(X_valid_90, y_valid_90_3d),
        epochs=200,
        batch_size=256,
        callbacks=[early_stop, reduce_lr],
        verbose=1,
    )


Dispositivo a usar: /device:GPU:0
Mixed precision: activado (mixed_float16)
[H60] Shapes OK: X_train (912, 29, 7) y_train (912, 29, 1) | X_valid (195, 29, 7) y_valid (195, 29, 1)
[H90] Shapes OK: X_train (912, 29, 7) y_train (912, 29, 1) | X_valid (195, 29, 7) y_valid (195, 29, 1)

FIT – Transformer H60 (VALID ONLY)
Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 46s 7s/step - loss: 2980.2109 - mean_absolute_error: 37.2069 - val_loss: 2504.4639 - val_mean_absolute_error: 36.3964 - learning_rate: 0.0010
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2839.5266 - mean_absolute_error: 36.6235 - val_loss: 2504.9539 - val_mean_absolute_error: 36.4524 - learning_rate: 0.0010
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2846.6758 - mean_absolute_error: 37.2646 - val_loss: 2487.5295 - val_mean_absolute_error: 36.3480 - learning_rate: 0.0010
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 2879.2095 - mean_absolute_error: 37.3287 - val_loss: 2463.0886 - val_mean_absolute_er

### **8.4. Predicción y evaluación (VALID ONLY)**

In [34]:
# ============================================================
# # Predicción genérica (Seq2Seq directo: TCN / GRU / LSTM)

# ============================================================
# Objetivo:
# - Predecir sobre VALID para H60 y H90
# - Convertir y_true/y_pred a 2D (N, H) para usar compute_seq2seq_metrics
# - Calcular métricas ML (MAE, RMSE, R2, DA_last si aplica)
# ============================================================

import numpy as np
import pandas as pd


# -------------------------
# Helpers de forma (2D / 3D)
# -------------------------
def to_2d_y(y: np.ndarray) -> np.ndarray:
    """
    Normaliza y a 2D:
      - (N, H, 1) -> (N, H)
      - (N, H)    -> (N, H)
    """
    if y.ndim == 3 and y.shape[-1] == 1:
        return y[..., 0]
    if y.ndim == 2:
        return y
    raise ValueError(f"Forma inesperada para y: {y.shape}")


# -------------------------
# Predicción genérica (RepeatVector: GRU/LSTM)
# -------------------------
def predict_seq2seq_2d(bundle_h: dict, model, *, split: str = "valid") -> tuple[np.ndarray, np.ndarray]:
    """
    Predicción para modelos RepeatVector (un solo input X).
    Retorna:
      - y_true_2d: (N, H)
      - y_pred_2d: (N, H)
    """
    # X del split
    X = bundle_h[split]["X"]

    # y verdadero desde el bundle (puede venir 2D o 3D)
    y_true_2d = to_2d_y(bundle_h[split]["y"])

    # Predicción del modelo:
    # - LSTM RepeatVector entrega (N, H, 1)
    y_pred_3d = model.predict(X, verbose=0)

    # Convertimos a 2D (N, H) para métricas
    y_pred_2d = to_2d_y(y_pred_3d)

    return y_true_2d, y_pred_2d

In [35]:
def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    """
    Convierte dict de métricas a tabla (1 fila).
    """
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA_last": metrics.get("DA_last"),
    }])


### **8.5. Flujo completo para H60 y H90**

In [36]:
# -------------------------
# H60 (VALID ONLY)
# -------------------------
y_true_60, y_pred_60 = predict_seq2seq_2d(
    bundle_60,
    model_tr_60,
    split="valid",
)
ml_valid_60 = compute_seq2seq_metrics(
    y_true_60,
    y_pred_60,
    compute_r2=True,
)

# -------------------------
# H90 (VALID ONLY)
# -------------------------
y_true_90, y_pred_90 = predict_seq2seq_2d(
    bundle_90,
    model_tr_90,
    split="valid",
)
ml_valid_90 = compute_seq2seq_metrics(
    y_true_90,
    y_pred_90,
    compute_r2=True,
)

## **8. Métricas ML**

In [37]:
# -------------------------
# Tabla final (VALID ONLY)
# -------------------------
df_valid_60 = metrics_to_df(ml_valid_60, model="transformer", split="valid", horizon=60)
df_valid_90 = metrics_to_df(ml_valid_90, model="transformer", split="valid", horizon=90)

pd.concat([df_valid_60, df_valid_90], ignore_index=True)

,model,split,horizon_min,MAE,RMSE,R2,DA_last
0,transformer,valid,60,33.131355,45.888635,0.162098,0.505155
1,transformer,valid,90,52.199697,68.975337,0.053868,0.500000


## **10. Guardar artefactos para Stage_08**

In [40]:
def save_json(obj: Dict[str, Any], path: Path) -> None:
    """Guarda un diccionario como JSON, creando directorios si es necesario."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [41]:
from pathlib import Path
from typing import Any, Dict
import numpy as np

def save_evaluation_artifacts(
    *,
    out_dir: Path,
    model_name: str,
    horizon: int,
    ml_valid: Dict[str, Any],
    y_valid: np.ndarray | None = None,
    y_pred_valid: np.ndarray | None = None,
    save_preds: bool = True,
) -> None:
    """
    Guarda SOLO artefactos de VALID (coherente con el esquema del libro):
      - metrics_ml_valid.json
      - pred_valid.npz (opcional)

    No guarda nada de TEST.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Métricas ML (VALID)
    # -------------------------
    save_json(
        {
            "model": model_name,
            "horizon_min": horizon,
            "split": "valid",
            **ml_valid,
        },
        out_dir / "metrics_ml_valid.json",
    )

    # -------------------------
    # Predicciones OOS (VALID) - opcional
    # -------------------------
    if save_preds:
        if y_valid is None or y_pred_valid is None:
            raise ValueError("Si save_preds=True, debe pasar y_valid y y_pred_valid.")
        np.savez_compressed(
            out_dir / "pred_valid.npz",
            y_true=np.asarray(y_valid),
            y_pred=np.asarray(y_pred_valid),
        )

    print(f"OK - artefactos guardados en: {out_dir}")

In [42]:
OUT_DIR_60 = Path("artifacts/06_transformer/h60")
OUT_DIR_60 = DRIVE_DIR / OUT_DIR_60 #Solo para la notebook

save_evaluation_artifacts(
    out_dir=OUT_DIR_60,
    model_name="transformer_h60",
    horizon=60,
    ml_valid=ml_valid_60,
    y_valid=y_true_60,        # ← 2D seguro
    y_pred_valid=y_pred_60,   # ← 2D
    save_preds=False,
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/06_transformer/h60


In [44]:
OUT_DIR_90 = Path("artifacts/06_transformer/h90")
OUT_DIR_90 = DRIVE_DIR / OUT_DIR_90 #Solo para la notebook

save_evaluation_artifacts(
    out_dir=OUT_DIR_90,
    model_name="transformer_h90",
    horizon=90,
    ml_valid=ml_valid_90,
    y_valid=y_true_90,
    y_pred_valid=y_pred_90,
    save_preds=False,
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/06_transformer/h90


## **11. Resumen**

- **Métricas ML (test):** MAE, RMSE, DA_last, R2
- **Métricas económicas como filtro (test):** Precision, Opportunity_Recall, Coverage
- Artefactos guardados en `reports/stage_07/h{H}/<model_name>/`

- Desempeño similar a GRU/LSTM: el Transformer no muestra una mejora clara frente a los modelos recurrentes previos.

- R² bajo en ambos horizontes: la capacidad explicativa sigue siendo limitada, especialmente en H=90.

- Error creciente con el horizonte: consistente con la mayor incertidumbre al predecir a 90 minutos.

- DA cercano a 0.5: comportamiento prácticamente equivalente a azar en la dirección final.

Conclusión: el cuello de botella no parece ser la arquitectura, sino la formulación del problema (ventana, features o target).